# 🔥 Never-AI — Super-Trainer GPU « Personnalité, Humeurs Vivantes & Réseaux » (v2.1)

Ce notebook entraîne **votre propre architecture neuronale libre `ChronoMorph-LM` (`Sprout-MoE` — ~72M paramètres, 8 experts MoE, mémoire Titans, optimiseur Muon)** sur un GPU **NVIDIA T4 / L4 / A100** gratuit de Google Colab.

### ✨ Nouveautés v2.1 :
- **📱 Flow Réseaux Sociaux & Argot FR naturel** : entraînement sur un corpus contrastif style Discord / X / TikTok / SMS + vraies discussions humaines francophones (`Aya FR` + `OpenAssistant FR`) + culture générale (`Wikipedia FR`).
- **🎲 6 Humeurs Vivantes conditionnées dans les neurones (`<|context|>Humeur: ...`)** : `😎 chill`, `🔥 taquin`, `⚡ hype`, `🌶️ insolent`, `🧠 philosophe`, `💛 complice` — votre IA change réellement de caractère selon son humeur !
- **✅ Zéro Google Drive requis** (`~5 Mo` sur votre réseau pendant tout l'entraînement + `~70 Mo` au téléchargement final du `.zip`).

### Mode d'emploi en 3 clics :
1. Menu **Exécution** → **Modifier le type d'exécution** → sélectionnez **T4 GPU** → **Enregistrer**.
2. Cliquez sur **Exécution → Tout exécuter** (`Ctrl+F9`).
3. Importez le fichier **`never-ai-brain-sprout-moe.zip`** téléchargé via **⚡ Super-Cortex → 📥 Importer `.zip` Colab** !

In [ ]:
# 1️⃣ VÉRIFICATION GPU & COFFRE LOCAL COLAB (ZÉRO BLOCAGE GOOGLE DRIVE !)
import os
import torch

TRY_GOOGLE_DRIVE = False  # Laissez False pour éviter toute erreur 'credential propagation'
VAULT_DIR = '/content/NeverAI_Vault/checkpoints'
DRIVE_BACKUP_DIR = None

if TRY_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_BACKUP_DIR = '/content/drive/MyDrive/NeverAI_Vault/checkpoints'
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        print('☁️ Google Drive connecté en bonus :', DRIVE_BACKUP_DIR)
    except Exception as err:
        print(f'ℹ️ Google Drive ignoré ({err}) -> utilisation du SSD local Colab + export .zip direct !')

os.makedirs(VAULT_DIR, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_label = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU (Activez T4 GPU dans Exécution -> Modifier le type d\'exécution !)'
print(f'✅ Accélérateur actif : {gpu_label} | PyTorch {torch.__version__}')
print(f'✅ Coffre neuronal prêt : {VAULT_DIR}')

In [ ]:
# 2️⃣ CLONAGE NEVER-AI & INSTALLATION ULTRA-RAPIDE (~8 SECONDES SUR PYTHON 3.13)
import os, subprocess, sys

if not os.path.exists('/content/never-ai/.git'):
    subprocess.run([
        'git', 'clone', '-b', 'arena/403f5d7f-never-ai', '--depth', '1',
        'https://github.com/mouusieur/never-ai.git', '/content/never-ai'
    ], check=True)
else:
    subprocess.run(['git', '-C', '/content/never-ai', 'pull', '--ff-only'], check=False)

os.chdir('/content/never-ai')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'tokenizers', 'safetensors', 'sympy', 'pyyaml', 'tqdm', 'datasets'
], check=True)

if '/content/never-ai/src' not in sys.path:
    sys.path.insert(0, '/content/never-ai/src')
print('✅ Moteur Never-AI v2.1 (Personnalité & Humeurs Vivantes) prêt sur', os.getcwd())

In [ ]:
# 3️⃣ ASSEMBLAGE DU CORPUS : RÉSEAUX SOCIAUX + 6 HUMEURS + DIALOGUES HUMAINS FR + WIKIPEDIA FR
import tempfile
from datasets import load_dataset
from never_ai.core.special_tokens import BOS_TOKEN, CONTEXT_TOKEN, EOS_TOKEN, NEVER_TOKEN, USER_TOKEN
from never_ai.cortex.mood_engine import LimbicMoodEngine
from never_ai.data.curriculum import synthesize_bilingual_curriculum
from never_ai.data.social_persona_corpus import synthesize_social_mood_corpus

N_WIKI_ARTICLES = 5500
N_HUMAN_DIALOGUES = 4500

raw_texts_for_tokenizer = []
formatted_docs = []
mood_tagger = LimbicMoodEngine(state_path=os.path.join(tempfile.gettempdir(), 'colab_mood_tagger.json'))

# A) Corpus Réseaux Sociaux, Argot FR naturel & 6 Humeurs Contrastives (chill, taquin, hype, insolent, philosophe, complice)
print('📱 Génération du corpus Réseaux Sociaux & 6 Humeurs Vivantes...')
soc_train, soc_val = synthesize_social_mood_corpus(seed=42, scale_factor=14)
formatted_docs.extend(soc_train)
raw_texts_for_tokenizer.extend(soc_train[:600])
print(f'   -> {len(soc_train):,} séquences multi-humeurs (Discord / X / TikTok / débats francs) prêtes !')

# B) Dialogues humains francophones spontanés (Cohere Aya FR + OpenAssistant OASST2 FR) étiquetés par humeur
human_pairs = 0
print('💬 Streaming de vraies conversations humaines en français (Aya FR + OpenAssistant)...')
try:
    ds_aya = load_dataset('CohereForAI/aya_dataset', split='train', streaming=True)
    for row in ds_aya:
        if row.get('language') != 'French':
            continue
        q_txt = (row.get('inputs') or '').strip()
        a_txt = (row.get('targets') or '').strip()
        if len(q_txt) > 8 and len(a_txt) > 15:
            m_id = mood_tagger.evolve_on_message(q_txt + ' ' + a_txt[:160])['current_mood']
            raw_texts_for_tokenizer.append(q_txt + '\n' + a_txt[:2000])
            formatted_docs.append(f'{BOS_TOKEN}{CONTEXT_TOKEN}Humeur: {m_id}{USER_TOKEN}{q_txt}{NEVER_TOKEN}{a_txt[:2200]}{EOS_TOKEN}')
            human_pairs += 1
            if human_pairs >= N_HUMAN_DIALOGUES // 2:
                break
except Exception as e:
    print(f'   -> Info Aya FR ({e})')

try:
    ds_oasst = load_dataset('OpenAssistant/oasst2', split='train', streaming=True)
    by_id = {}
    for row in ds_oasst:
        if row.get('lang') != 'fr':
            continue
        msg_id, parent_id, role = row.get('message_id'), row.get('parent_id'), row.get('role')
        txt = (row.get('text') or '').strip()
        if not txt:
            continue
        by_id[msg_id] = (role, txt)
        if role == 'assistant' and parent_id in by_id and by_id[parent_id][0] == 'prompter':
            q_txt = by_id[parent_id][1]
            m_id = mood_tagger.evolve_on_message(q_txt + ' ' + txt[:160])['current_mood']
            raw_texts_for_tokenizer.append(q_txt + '\n' + txt)
            formatted_docs.append(f'{BOS_TOKEN}{CONTEXT_TOKEN}Humeur: {m_id}{USER_TOKEN}{q_txt}{NEVER_TOKEN}{txt}{EOS_TOKEN}')
            human_pairs += 1
            if human_pairs >= N_HUMAN_DIALOGUES:
                break
except Exception as e:
    print(f'   -> Info OASST2 ({e})')
print(f'   -> {human_pairs:,} échanges humains naturels chargés et classés par humeur !')

# C) Culture générale & fluidité française (Wikipedia FR)
print(f'📚 Streaming de {N_WIKI_ARTICLES} articles riches de Wikipedia FR...')
ds_wiki = load_dataset('wikimedia/wikipedia', '20231101.fr', split='train', streaming=True)
wiki_count = 0
for row in ds_wiki:
    txt = (row.get('text') or '').strip()
    title = (row.get('title') or '').strip()
    if len(txt) > 500:
        excerpt = txt[:3200]
        raw_texts_for_tokenizer.append(excerpt)
        if wiki_count % 2 == 0 and title:
            formatted_docs.append(
                f'{BOS_TOKEN}{CONTEXT_TOKEN}Humeur: chill{USER_TOKEN}Dis-moi ce que tu sais sur {title}.{NEVER_TOKEN}{excerpt[:1800]}{EOS_TOKEN}'
            )
        else:
            formatted_docs.append(f'{BOS_TOKEN}{excerpt}{EOS_TOKEN}')
        wiki_count += 1
        if wiki_count >= N_WIKI_ARTICLES:
            break

# D) Raisonnement logique, maths & code Python
cur_train, cur_val = synthesize_bilingual_curriculum(seed=42, scale_factor=3)
formatted_docs.extend(cur_train)
raw_texts_for_tokenizer.extend(cur_train[:250])

total_chars = sum(len(t) for t in formatted_docs)
print(f'✅ Corpus complet assemblé : {len(formatted_docs):,} séquences ({total_chars / 1e6:.1f} Mo de personnalité, argot FR & savoir)')

In [ ]:
# 4️⃣ ENTRAÎNEMENT GPU INTENSIF : CHRONOMORPH SPROUT-MOE (~72M PARAMS + MUON + WSM)
import random, shutil, time
import torch
from never_ai.core.config import ChronoMorphConfig
from never_ai.core.model import ChronoMorphLM
from never_ai.core.tokenizer import MorphTokenizer
from never_ai.data.curriculum import ContinualDatasetAndReplay
from never_ai.optim.composite_muon import CompositeMuonOptimizer, WSMCheckpointMerger
from never_ai.vault.manager import VaultManager

# 1500 steps ≈ 6-8 min sur T4 GPU (montez à 5000 steps ≈ 22 min pour une plume encore plus fine !)
STEPS = 1500
BATCH_SIZE = 12 if device == 'cuda' else 2
SEQ_LEN = 384 if device == 'cuda' else 128

print('🔤 Entraînement du tokenizer BPE Byte-Level (vocab=8192) sur le français vivant & réseaux...')
tok = MorphTokenizer.train_from_iterator(raw_texts_for_tokenizer, vocab_size=8192)

cfg = ChronoMorphConfig.sprout_moe(vocab_size=tok.vocab_size)
model = ChronoMorphLM(cfg).to(device)
counts = model.count_parameters()
print(f"🧠 Modèle {cfg.stage_name} sur {device.upper()} : {counts['total_params']/1e6:.2f}M paramètres totaux ({counts['active_params']/1e6:.2f}M actifs/token)")

rng = random.Random(42)
rng.shuffle(formatted_docs)
split_idx = max(16, int(len(formatted_docs) * 0.97))
dataset = ContinualDatasetAndReplay(tok, seq_len=SEQ_LEN)
stats_ds = dataset.load_corpora(formatted_docs[:split_idx], formatted_docs[split_idx:] + soc_val + cur_val)
print(f"📦 Séquences empaquetées (longueur {SEQ_LEN}) : {stats_ds['train_chunks']:,} train | {stats_ds['val_chunks']:,} val")

opt = CompositeMuonOptimizer(model, muon_lr=3.0e-3, adamw_lr=8.0e-4, weight_decay=0.01, warmup_steps=80)
wsm = WSMCheckpointMerger(max_snapshots=6)
vault = VaultManager(vault_root=VAULT_DIR)
amp_dtype = torch.bfloat16 if (device == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float32

t0 = time.time()
tokens_seen = 0
model.train()
for step in range(1, STEPS + 1):
    batch = dataset.sample_batch(batch_size=BATCH_SIZE, split='train', replay_ratio=0.20, device=device)
    opt.zero_grad()
    with torch.amp.autocast(device_type=device, dtype=amp_dtype, enabled=(amp_dtype != torch.float32)):
        out = model(batch.input_ids, targets=batch.targets, loss_mask=batch.loss_mask, update_titans=True)
        loss = out.loss
    loss.backward()
    opt.step(model)
    model.update_moe_biases()
    tokens_seen += int(batch.input_ids.numel())

    if step % 100 == 0 or step == 1:
        dt = max(1e-3, time.time() - t0)
        tok_s = tokens_seen / dt
        print(f'⚡ Step {step:4d}/{STEPS} | CE Loss = {out.ce_loss:.4f} | Vitesse = {tok_s:,.0f} tok/s | Tokens = {tokens_seen:,}')

    if step % 300 == 0 or step == STEPS:
        wsm.record_snapshot(model)
        consolidated = wsm.build_consolidated_state_dict(fallback_model=model)
        ckpt_path = vault.save_checkpoint(
            'latest',
            model,
            tok,
            metrics={
                'val_ce_loss': round(float(out.ce_loss), 4),
                'step': step,
                'total_tokens_trained': tokens_seen,
                'colab_gpu_trained': True,
                'mood_conditioned': True,
            },
            consolidated_state_dict=consolidated,
        )
        if DRIVE_BACKUP_DIR:
            shutil.copytree(ckpt_path, os.path.join(DRIVE_BACKUP_DIR, 'latest'), dirs_exist_ok=True)
        print(f'💾 Checkpoint WSM sauvegardé -> {ckpt_path}')

In [ ]:
# 5️⃣ TEST MULTI-HUMEURS EN DIRECT SUR LE GPU + TÉLÉCHARGEMENT .ZIP POUR NEVER-AI !
import os, zipfile
from never_ai.core.special_tokens import BOS_TOKEN, CONTEXT_TOKEN, EOS_ID, NEVER_TOKEN, USER_ID, USER_TOKEN

model.eval()
stop_ids = {EOS_ID, USER_ID}
same_prompt = "wsh ça dit quoi ? t'es de quelle humeur aujourd'hui ?"

print('🎭 Test de la MÊME question selon 3 humeurs différentes de votre IA :\n' + '=' * 70)
for mood_id, emoji in [('chill', '😎'), ('taquin', '🔥'), ('philosophe', '🧠')]:
    p_ids = tok.encode(f'{BOS_TOKEN}{CONTEXT_TOKEN}Humeur: {mood_id}{USER_TOKEN}{same_prompt}{NEVER_TOKEN}')
    gen_ids, _ = model.generate(p_ids, max_new_tokens=90, temperature=0.65, top_p=0.9, stop_token_ids=stop_ids)
    reply = tok.decode(gen_ids[len(p_ids):], skip_special_tokens=True).strip()
    print(f'{emoji} [Humeur = {mood_id}] : {reply}\n')

zip_out = '/content/never-ai-brain-sprout-moe.zip'
latest_dir = os.path.join(VAULT_DIR, 'latest')
with zipfile.ZipFile(zip_out, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(latest_dir)):
        fpath = os.path.join(latest_dir, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, arcname=fname)

size_mb = os.path.getsize(zip_out) / (1024 * 1024)
print(f'📦 Archive prête : {zip_out} ({size_mb:.1f} Mo)')
print('👉 Dans Never-AI Studio : cliquez sur ⚡ Super-Cortex -> 📥 Importer .zip Colab !')

try:
    from google.colab import files
    files.download(zip_out)
except Exception as e:
    print(f'ℹ️ Téléchargement disponible dans le panneau Fichiers (icône 📁 à gauche) : {zip_out} ({e})')